In [1]:
# Dati di base

# Motore
C0 = 15 # Nm
km = 0.003 # Nm/rpm
C = lambda omega: C0 - km*omega

# Utilizzatore
Cu0 = 100 # Nm
ku = 0.001 # nm/rpm^2
Cu = lambda omega: Cu0 + ku*omega**2

# riduttore
tau = 1/11
eta = 0.91

# Freno
ri = 100*6 # mm
re = 150*6 # mm
Fd = 1000 # N
mu = 0.3 # attrito
Mf = mu*Fd*(re+ri)*0.5

# inerzie in kg m^2
Jd = 0.2
Jm = 0.1
Jv = 3.5 

# inerzia equivalente totale motore flusso potenza diretto
Jeq_d = eta*(Jm + Jd) + tau**2*Jv
Jeq_r = (Jd) + eta*tau**2*Jv

# Materiale ingranaggi 
Su = 700
Sn = 500

In [2]:
omega_regime_max = C0/km # rpm
print(f"Massima coppia teorica erogabile utilizzatore: {C0/tau} Nm")
print(f"Massima velocità di regime: {omega_regime_max} rpm motore, {omega_regime_max*tau:.2f} rpm utilizzatore")
print(f"Massima coppia erogabile da utilizzatore a regime teorico (ipotizzando regime massimo motore): {Cu(omega_regime_max*tau):.2f}")
print(f"Momento frenante: {Mf:.2f} N mm")

Massima coppia teorica erogabile utilizzatore: 165.0 Nm
Massima velocità di regime: 5000.0 rpm motore, 454.55 rpm utilizzatore
Massima coppia erogabile da utilizzatore a regime teorico (ipotizzando regime massimo motore): 306.61
Momento frenante: 225000.00 N mm


In [3]:
# calcolo velocità di regime
import numpy as np

omega_m_regime = (-eta*km + np.sqrt((eta*km)**2 + 4* (eta*C0 - Cu0*tau)*ku*tau**3))/2/ku/tau**3
Cm_regime = C(omega_m_regime)
Cu_regime = Cu(omega_m_regime*tau)
print(f"Velocità di regime effettiva: {omega_m_regime:.2f} motore, {omega_m_regime*tau:.2f} utilizzatore")
print(f"Coppie a regime: {Cm_regime:.2f} Nm erogata motore, {Cu_regime:.2f} Nm assorbita utilizzatore")


Velocità di regime effettiva: 1244.06 motore, 113.10 utilizzatore
Coppie a regime: 11.27 Nm erogata motore, 112.79 Nm assorbita utilizzatore


In [4]:
# calcolo accelerazione di spunto
omega_dot_m = (-Cu0*tau + eta*C0)/Jeq_d
print(f"Coppia motore allo spunto: {C0:.2f} Nm")
print(f"Coppia utilizzatore assorbita allo spunto: {Cu0:.2f} Nm")
print(f"Accelerazione motore allo spunto: {omega_dot_m:.2f} rad/s^2")
print(f"Coppia lato motore allo spunto: {C0-omega_dot_m*Jeq_d:.2f} Nm")
print(f"Coppia lato utilizzatore allo spunto: {(Cu0 + omega_dot_m*tau*Jv):.2f} Nm")


Coppia motore allo spunto: 15.00 Nm
Coppia utilizzatore assorbita allo spunto: 100.00 Nm
Accelerazione motore allo spunto: 15.10 rad/s^2
Coppia lato motore allo spunto: 10.44 Nm
Coppia lato utilizzatore allo spunto: 104.80 Nm


In [5]:
# Calcolo accelerazione durante la frenata di emergenza
Jeq_r = (Jd) + eta*tau**2*(Jv)
Mf_Nm = Mf/1000
omega_dot_m_brake = (-Mf_Nm + eta*tau*(-Cu_regime))/Jeq_r
Cu_brake = -Cu_regime - tau*omega_dot_m_brake*(Jv)
print(f"Coppia frenante allo spunto: {Mf_Nm:.2f} Nm")
print(f"Coppia utilizzatore assorbita durante la frenata: {Cu_regime:.2f} Nm")
print(f"Accelerazione motore durante la frenata: {omega_dot_m_brake:.2f} rad/s^2")
print(f"Coppia lato motore: {Mf_Nm + omega_dot_m_brake*Jd}")
print(f"Coppia lato utilizzatore: {Cu_brake:.2f} Nm")
print(f"Tempo di arresto: {-omega_m_regime/omega_dot_m_brake:.2f} secondi")


Coppia frenante allo spunto: 225.00 Nm
Coppia utilizzatore assorbita durante la frenata: 112.79 Nm
Accelerazione motore durante la frenata: -1035.39 rad/s^2
Coppia lato motore: 17.922869076554548
Coppia lato utilizzatore: 216.65 Nm
Tempo di arresto: 1.20 secondi


In [6]:
# Dimensionamento di massima del modulo
m_allowable = [0.5, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 10]
Sy = 700*0.5 # Mpa
C_gear = Cu_regime*1000 # Nmm coppia regime per un pseudo coefficiente di sicurezza
print(f"Coppia: {C_gear:.2f} Nmm")
nZ_p = 17
nZ_c = 56
tau_stage = nZ_p/nZ_c
tau_total = tau_stage*tau_stage
tau_desired = tau
print(f"Rapporto di trasmissione: {tau_total:.4f}")
print(f"Rapporto di trasmissione desiderato: {tau_desired:.4f}")
print(f"Scostamento: {abs(tau_total - tau_desired)/tau_desired*100:.2f} %")
Y = 0.3 # Lewis geometric factor
m = (2*C_gear/(10*Sy*nZ_c*Y))**(1/3)
print(f"Calcolo del modulo teorico: {m:.2f}")


# get the closest allowable module, rounded up
m = min([x for x in m_allowable if x >= m])
m = 2.5
print(f"Estimation of module: {m:.2f}")
Dp = m*nZ_p
Dc = m*nZ_c
Ft = 2*C_gear/Dc
print(f"Diametro pignone: {Dp:.2f}")
print(f"Diametro corona: {Dc:.2f}")
print(f"Ingombro a: {Dp+Dc:.2f}")
print(f"Ingombro b: {Dc:.2f}")
print(f"Ingombro c: {m*10*5:.2f}")
print(f"Forza tangenziale: {Ft:.2f} N")
Mc = (Dc/2)**2*np.pi*10*m/1000/1000/1000*7800 # massa corona
print(f"Momento di inerzia ingranaggio: {0.5*Mc*(Dc/2/1000)**2:.2f} kg m^2")
vel_t = omega_m_regime*tau*np.pi*2/60*Dc/2/1000
print(f"Velocità tangenziale: {vel_t:.2f} m/s")

Coppia: 112790.82 Nmm
Rapporto di trasmissione: 0.0922
Rapporto di trasmissione desiderato: 0.0909
Scostamento: 1.37 %
Calcolo del modulo teorico: 1.57
Estimation of module: 2.50
Diametro pignone: 42.50
Diametro corona: 140.00
Ingombro a: 182.50
Ingombro b: 140.00
Ingombro c: 125.00
Forza tangenziale: 1611.30 N
Momento di inerzia ingranaggio: 0.01 kg m^2
Velocità tangenziale: 0.83 m/s


In [ ]:
# Verifica a bending fatigue
J = 0.28 # fattore di forma
Kv = 1.5
Ko = 1.25
Km = 1.3
sigma_bending = Ft/m/10/m/J*Kv*Ko*Km
print(f"Tensione massima a fondo dente: {sigma_bending:.2f} MPa")

Sn= 0.5*Su*1*0.85*0.9*0.814*1*1.4
print(f"Tensione ammissibile per vita infinita: {Sn} MPa")

C_brake = Cu_brake*1000 
Ft_brake = 2*C_brake/Dc

sigma_braking = Ft_brake/m/10/m/J*Kv*Ko*Km
print(f"Maximum stress during braking: {sigma_braking:.2f} MPa")

Tensione massima a fondo dente: 224.43 MPa
Tensione ammissibile per vita infinita: 305.12789999999995 MPa
Maximum stress during braking: 431.09


In [8]:
# Verifica a fatica pitting
Sfe = 1700 # Mpa
ni = 0.3# Poisson
E = 210000 # Mpa
Cp = 0.564*np.sqrt(1/(2*(1-ni**2)/E))
print(f"Costante del materiale: {Cp:.2f}")
R = Dc/Dp
I = np.sin(20*np.pi/180)*np.cos(20*np.pi/180)*R/(1+R)/2
sigmaH = Cp*np.sqrt(Ft/10/m/Dp/I*1.8*1.25**1.5)
print(f"Pressione di contatto: {sigmaH:.2f} MPa")

ore_funzionamento = 50000 # ore
minuti_funzionamento = ore_funzionamento*60
print(f"Minuti di funzionamento: {minuti_funzionamento*60}")
print(f"Cicli di lavoro: {minuti_funzionamento*omega_m_regime/tau_stage:.2f}")
Cli = 0.7
print(f"Coefficiente di sicurezza a fatica: {Sfe*Cli/sigmaH:.2f}")

Costante del materiale: 191.58
Pressione di contatto: 1065.76 MPa
Minuti di funzionamento: 180000000
Cicli di lavoro: 12294257722.77
Coefficiente di sicurezza a fatica: 1.12
